In [1]:
print("hello world")

hello world


In [3]:
from googleapiclient.discovery import build
import pymongo
import psycopg2
import pandas as pd
import streamlit as st


In [4]:
#API KEY Connection
def api_connect():
    api_id="Enter the API key from googleclient"

    api_service_code="youtube"
    api_version="v3"

    youtube=build(api_service_code,api_version, developerKey=api_id)

    return youtube

youtube=api_connect()

In [5]:
#channel information
def get_Channel_info(Channel_id):

    request= youtube.channels().list(
        part="Snippet,ContentDetails,Statistics",
        id=Channel_id
    )
    response=request.execute()

    for i in response["items"]:
        data=dict(channel_name=i["snippet"]["title"],
                channel_Id=i["id"],
                subscribers=i['statistics']["subscriberCount"],
                Views=i["statistics"]["viewCount"],
                Total_Videos=i["statistics"]["videoCount"],
                Channel_Description=i["snippet"]["description"],
                PlayList_Id=i["contentDetails"]["relatedPlaylists"]["uploads"] )

    return data

In [6]:
Channel_details= get_Channel_info("UCY6KjrDBN_tIRFT_QNqQbRQ")

Channel_details

{'channel_name': 'Madan Gowri',
 'channel_Id': 'UCY6KjrDBN_tIRFT_QNqQbRQ',
 'subscribers': '8820000',
 'Views': '2947675782',
 'Total_Videos': '4323',
 'Channel_Description': 'MG Squad 🖖🏻 🇮🇳',
 'PlayList_Id': 'UUY6KjrDBN_tIRFT_QNqQbRQ'}

In [7]:
def get_video_ids(channel_id):

    video_ids=[]


    response=youtube.channels().list(id=channel_id,
                                        part='contentDetails').execute()
    playlist_Id=response['items'][0]['contentDetails']['relatedPlaylists']['uploads']

    next_page_token=None

    while True:

        response1=youtube.playlistItems().list(
                                        part='snippet',
                                        playlistId=playlist_Id,
                                         maxResults=50,
                                         pageToken=next_page_token).execute()

        for i in range(len(response1['items'])):
            video_ids.append(response1['items'][i]['snippet']['resourceId']['videoId'])

        next_page_token=response1.get('nextPageToken')

        if next_page_token is None:
            break

        return video_ids

In [8]:
video_Ids = get_video_ids('UCY6KjrDBN_tIRFT_QNqQbRQ')

In [9]:
len(video_Ids)

50

In [10]:
#video information
def get_video_info(video_ids):
    video_data=[]
    for video_id in video_ids:
        request=youtube.videos().list(
            part="snippet,contentDetails,statistics",
            id=video_id

        )
        response=request.execute()

        for item in response["items"]:
            data=dict(channel_Name=item['snippet']['channelTitle'],
                    channel_Id=item['snippet']['channelTitle'],
                    video_Id=item['id'],
                    Title=item['snippet']['title'],
                    Tags=item['snippet'].get('tags'),
                    Thumbnail=item['snippet']['thumbnails']['default']['url'],
                    Description=item['snippet'].get('description'),
                    PublishedAt=item['snippet']['publishedAt'],
                    Duration=item['contentDetails']['duration'] , 
                    Views=item['statistics'].get('viewCount'),
                    Likes=item['statistics'].get('likeCount'),
                    Comments=item['statistics'].get('commentCount'),
                    Favorite_Count=item['statistics']['favoriteCount'],
                    Definition=item['contentDetails']['definition'],
                    Caption_Status=item['contentDetails']['caption']
                    )

            video_data.append(data) 
    return video_data

In [11]:
video_Ids = get_video_ids('UCY6KjrDBN_tIRFT_QNqQbRQ')

video_data=[]
for video_id in video_Ids:
    request=youtube.videos().list(
        part="snippet,contentDetails,statistics",
        id=video_id

    )
    response=request.execute()

In [12]:
video_details=get_video_info(video_Ids)

In [13]:
#comment information
def get_comment_info(video_ids):

    Comment_data=[]
    try:
        for video_id in video_ids:
            request=youtube.commentThreads().list(
                part="snippet",
                videoId=video_id,
                maxResults=50
                    )
            response=request.execute()

            for item in response["items"]:
                data=dict(comment_Id=item["snippet"][ 'topLevelComment']["id"],
                        video_Id=item["snippet"]['topLevelComment']["snippet"][ 'videoId'],
                        Comment_Text=item["snippet"]['topLevelComment']["snippet"]['textDisplay'],
                        Comment_Author=item["snippet"]['topLevelComment']["snippet"]['authorDisplayName'],
                        Comment_Published=item["snippet"]['topLevelComment']["snippet"]['publishedAt'])

                Comment_data.append(data)       
    except:
        pass
    return Comment_data

In [14]:
Comment_details=get_comment_info(video_Ids)

In [15]:
#playlist_details

def get_playlist_details(channel_id):
        next_page_token=None
        All_data=[]
        while True:
                request = youtube.playlists().list(
                                        part='snippet,contentDetails',
                                        channelId=channel_id,
                                        maxResults=50,
                                        pageToken=next_page_token)
                response=request.execute()

                for item in response['items']:
                        data=dict(playlist_Id=item["id"],
                                Title=item['snippet']['title'],
                                Channel_Id=item['snippet']['channelId'],
                                Channel_Name=item['snippet']['channelTitle'],
                                PublishedAt=item['snippet']['publishedAt'],
                                Video_Count=item['contentDetails']['itemCount'])

                        All_data.append(data)   
                next_page_token=response.get('nextPageToken')  
                if  next_page_token is None:
                        break
        return All_data

In [16]:
playlist_details= get_playlist_details('UCY6KjrDBN_tIRFT_QNqQbRQ')


In [17]:
playlist_details=get_playlist_details("UCBR8-60-B28hp2BmDPdntcQ")

In [18]:
playlist_details=get_playlist_details("UCvrhwpnp2DHYQ1CbXby9ypQ")

In [19]:
playlist_details=get_playlist_details("UCMoJmRNcc-n8WdesyjQ6i_g")

In [31]:
client=pymongo.MongoClient("mongodb+srv://<username>:<password>@cluster0.fausrjk.mongodb.net/?retryWrites=true&w=majority&appName=Cluster0")
db= client["youtube_data"]
#Collection=db["channel_details"]
#x={"name":"leo","year":2023}
#Collection.insert_one(x)

In [32]:
all_channels=[]
db=client["youtube_data"]
coll1=db["channel_details"]
for ch_data in coll1.find({},{"_id":0,"channel_information":1}):
    all_channels.append(ch_data['channel_information']['channel_name'])
   #ch_list.append(ch_data["channel_information"])
#df=pd.DataFrame(ch_list)

In [33]:
single_channel_details

NameError: name 'single_channel_details' is not defined

In [34]:
single_channel_details=[]
db = client["youtube_data"]
coll1 = db["channel_details"]
for ch_data in coll1.find({"channel_information.channel_name": "Madan Gowri"}, {"_id": 0,}):
    single_channel_details.append(ch_data["channel_information"])
df_single_channel_details=pd.DataFrame(single_channel_details)

In [35]:
df_single_channel_details

""


In [36]:
def channel_details(channel_id):
    ch_details= get_Channel_info(channel_id)
    pl_details= get_playlist_details(channel_id)
    vi_ids=get_video_ids(channel_id)
    vi_details=get_video_info(vi_ids)
    com_details=get_comment_info(vi_ids)

    coll1=db["channel_details"]
    coll1.insert_one({"channel_information":ch_details,"playlist_information":pl_details,
                      "video_information":vi_details,"comment_information":com_details})
    
    return "upload completed"
    

In [ ]:
#insert=channel_details("UCY6KjrDBN_tIRFT_QNqQbRQ")#madhan gowri

In [ ]:
#insert=channel_details("UCBR8-60-B28hp2BmDPdntcQ")#youtube

In [ ]:
#insert=channel_details("UCvrhwpnp2DHYQ1CbXby9ypQ")#vijay tele

In [ ]:
#insert=channel_details("UCMoJmRNcc-n8WdesyjQ6i_g")#oppo

In [38]:
def channel_table(channel_name_s):

    mydb = psycopg2.connect(
        host="localhost",
        user="postgres",
        password="password",
            database="databasename",
        port="5432",
    )
    cursor = mydb.cursor()


    create_query = """create table if not exists channels(channel_name varchar(100),
                                                        channel_Id varchar(80) primary key,
                                                        subscribers bigint,
                                                        Views bigint,
                                                        Total_Videos int,
                                                        Channel_Description text,
                                                        PlayList_Id varchar(80))"""

    cursor.execute(create_query)
    mydb.commit()


    single_channel_details=[]
    db = client["youtube_data"]
    coll1 = db["channel_details"]
    for ch_data in coll1.find({"channel_information.channel_name": channel_name_s}, {"_id": 0,}):
        single_channel_details.append(ch_data["channel_information"])
    df_single_channel_details=pd.DataFrame(single_channel_details)


    for index, row in df_single_channel_details.iterrows():
        insert_query = """insert into channels(channel_name,
                                                channel_Id,
                                                subscribers,
                                                Views,
                                                Total_Videos,
                                                Channel_Description,
                                                PlayList_Id)
                                                
                                                
                                                values(%s,%s,%s,%s,%s,%s,%s)"""
        values = (
            row["channel_name"],
            row["channel_Id"],
            row["subscribers"],
            row["Views"],
            row["Total_Videos"],
            row["Channel_Description"],
            row["PlayList_Id"],
        )
        try:
            cursor.execute(insert_query, values)
            mydb.commit()
        except:
            print("channels values are already inserted")

In [39]:
channel_table("Priyanka Deshpande")

channels values are already inserted


In [41]:
df

NameError: name 'df' is not defined

In [48]:

single_playlist_details=[]
db = client["youtube_data"]
coll1 = db["channel_details"]
for ch_data in coll1.find({"channel_information.channel_name": "Vj Siddhu Vlogs"}, {"_id": 0,}):
       single_playlist_details.append(ch_data["playlist_information"])

df_single_playlist_details=pd.DataFrame(single_playlist_details[0])


In [49]:
# playlist table
def playlist_table(channel_name_s):
        mydb = psycopg2.connect(
            host="localhost",
            user="postgres",
            password="password",
            database="databasename",
            port="5432",
        )
        cursor = mydb.cursor()


        create_query = """create table if not exists playlists(playlist_Id varchar(100) primary key,
                                                        Title varchar(100),
                                                        Channel_Id varchar(100),
                                                        Channel_Name varchar(100),
                                                        PublishedAt timestamp,
                                                        Video_Count int
                                                        )"""

        cursor.execute(create_query)
        mydb.commit()

    
        single_playlist_details=[]
        db = client["youtube_data"]
        coll1 = db["channel_details"]
        for ch_data in coll1.find({"channel_information.channel_name": channel_name_s}, {"_id": 0,}):
                single_playlist_details.append(ch_data["playlist_information"])

        df_single_playlist_details=pd.DataFrame(single_playlist_details[0])


        for index, row in df_single_playlist_details.iterrows():
            insert_query = """insert into playlists(playlist_Id,
                                                Title,
                                                Channel_Id,
                                                Channel_Name,
                                                PublishedAt,
                                                Video_Count
                                                )
                                                
                                                values(%s,%s,%s,%s,%s,%s)"""
            values = (
                row["playlist_Id"],
                row["Title"],
                row["Channel_Id"],
                row["Channel_Name"],
                row["PublishedAt"],
                row["Video_Count"],
            )
            cursor.execute(insert_query, values)
            mydb.commit()

In [51]:
single_video_details=[]
db = client["youtube_data"]
coll1 = db["channel_details"]
for ch_data in coll1.find({"channel_information.channel_name": "Vj Siddhu Vlogs"}, {"_id": 0,}):
        single_video_details.append(ch_data["video_information"])

df_single_video_details=pd.DataFrame(single_video_details[0])



In [52]:
df_single_video_details

,channel_Name,channel_Id,video_Id,Title,Tags,Thumbnail,Description,PublishedAt,Duration,Views,Likes,Comments,Favorite_Count,Definition,Caption_Status
0,Vj Siddhu Vlogs,Vj Siddhu Vlogs,Bi_Ot8_yDpo,வித விதமா ரக ரகமா Shopping 😜 |South Korea Ep-7...,"[Vj Siddhu Vlogs, Vj Siddhu Vlogs Youtube chan...",https://i.ytimg.com/vi/Bi_Ot8_yDpo/default.jpg,For Business inquiries please contact us :7200...,2024-05-22T09:50:07Z,PT18M50S,632051,86866,2821,0,hd,false
1,Vj Siddhu Vlogs,Vj Siddhu Vlogs,bZvlJUkprrg,Sports day நடத்தப்போறோம் 🥳| Vj Siddhu Vlogs,"[Vj Siddhu Vlogs, Vj Siddhu Vlogs Youtube chan...",https://i.ytimg.com/vi/bZvlJUkprrg/default.jpg,For Business inquiries please contact us :7200...,2024-05-21T09:30:09Z,PT29M33S,1640552,127846,2409,0,hd,false
2,Vj Siddhu Vlogs,Vj Siddhu Vlogs,cL9hOBbrk2A,Korean Convenience store visit | Vj Siddhu Vlogs,"[Vj Siddhu Vlogs, Vj Siddhu Vlogs Youtube chan...",https://i.ytimg.com/vi/cL9hOBbrk2A/default.jpg,For Business inquiries please contact us :7200...,2024-05-20T09:30:08Z,PT22M15S,1698772,125904,1572,0,hd,false
3,Vj Siddhu Vlogs,Vj Siddhu Vlogs,HdmDYVko7nI,என்னடா இதெல்லாம் திங்கிறானுங்க 🤢 🦐 | Korea Ep-...,"[Vj Siddhu Vlogs, Vj Siddhu Vlogs Youtube chan...",https://i.ytimg.com/vi/HdmDYVko7nI/default.jpg,For Business inquiries please contact us :7200...,2024-05-19T04:30:08Z,PT18M22S,1691446,134832,1444,0,hd,true
4,Vj Siddhu Vlogs,Vj Siddhu Vlogs,wjrMQnwEiUo,Kpop vs Hiphop⚔️👊 | BTS | South Korea Ep-4 | ...,"[Vj Siddhu Vlogs, Vj Siddhu Vlogs Youtube chan...",https://i.ytimg.com/vi/wjrMQnwEiUo/default.jpg,For Business inquiries please contact us :7200...,2024-05-17T09:30:08Z,PT16M58S,1691647,131676,3695,0,hd,true
5,Vj Siddhu Vlogs,Vj Siddhu Vlogs,3uugSgVkcxA,BTS அ பாக்க போறோம்😍 | South Korea Ep-3 | Vj Si...,"[Vj Siddhu Vlogs, Vj Siddhu Vlogs Youtube chan...",https://i.ytimg.com/vi/3uugSgVkcxA/default.jpg,For Business inquiries please contact us :7200...,2024-05-15T09:30:08Z,PT19M9S,2239205,165360,6146,0,hd,true
6,Vj Siddhu Vlogs,Vj Siddhu Vlogs,AMbzmzDxJP4,Automatic toilet ஆ ? 😱 | South Korea Ep-2 | Vj...,"[Vj Siddhu Vlogs, Vj Siddhu Vlogs Youtube chan...",https://i.ytimg.com/vi/AMbzmzDxJP4/default.jpg,For Business inquiries please contact us :7200...,2024-05-13T09:30:08Z,PT13M57S,2148915,170942,2455,0,hd,true
7,Vj Siddhu Vlogs,Vj Siddhu Vlogs,M9hbj3phdT8,மட்ட மதியானத்துல இப்படி குளுருதே🥶 | South Kore...,"[Vj Siddhu Vlogs, Vj Siddhu Vlogs Youtube chan...",https://i.ytimg.com/vi/M9hbj3phdT8/default.jpg,GT Holidays:\nContact GT holidays for Tour Pac...,2024-05-12T04:30:08Z,PT17M47S,2579937,173670,3009,0,hd,true
8,Vj Siddhu Vlogs,Vj Siddhu Vlogs,c-GfFMdwofE,தம்பி வாசிக்க அண்ணன் பாட ஒரே கூத்து தான்😂💥 | P...,"[Vj Siddhu Vlogs, Vj Siddhu Vlogs Youtube chan...",https://i.ytimg.com/vi/c-GfFMdwofE/default.jpg,For Business inquiries please contact us :7200...,2024-05-10T09:30:08Z,PT25M15S,1870212,147560,1821,0,hd,true
9,Vj Siddhu Vlogs,Vj Siddhu Vlogs,_I4Zax5Wkwk,ஒரு Round போடுவோமா Kavin bro🥂 | Vj Siddhu Vlogs,"[Vj Siddhu Vlogs, Vj Siddhu Vlogs Youtube chan...",https://i.ytimg.com/vi/_I4Zax5Wkwk/default.jpg,For Business inquiries please contact us :7200...,2024-05-08T09:45:08Z,PT23M1S,2511698,178851,2478,0,hd,true


In [54]:
# video  table
def videos_table(channel_name_s):
    mydb = psycopg2.connect(
        host="localhost",
        user="postgres",
        password="password",
            database="databasename",
        port="5432",
    )
    cursor = mydb.cursor()

    
    create_query = """create table if not exists videos(Channel_Name varchar(100),
                                                      Channel_Id varchar(100),
                                                      video_Id varchar(30),
                                                      Title varchar(150),
                                                      Tags text,
                                                      Thumbnail varchar(200),
                                                      Description text,
                                                      Published_Date timestamp,
                                                      Duration interval,
                                                      Views bigint,
                                                      Likes bigint,
                                                      Comments int,
                                                      Favorite_Count int,
                                                      Definition varchar(10),
                                                      Caption_Status varchar(50)
                                                      )"""

    cursor.execute(create_query)
    mydb.commit()

    single_video_details=[]
    db = client["youtube_data"]
    coll1 = db["channel_details"]
    for ch_data in coll1.find({"channel_information.channel_name": channel_name_s}, {"_id": 0,}):
            single_video_details.append(ch_data["video_information"])

    df_single_video_details=pd.DataFrame(single_video_details[0])


    for index, row in df_single_video_details.iterrows():
        insert_query = """insert into videos(Channel_Name,
                                          Channel_Id,
                                          video_Id,
                                          Title,
                                          Tags,
                                          Thumbnail,
                                          Description,
                                          Published_Date,
                                          Duration,
                                          Views,
                                          Likes,
                                          Comments,
                                          Favorite_Count,
                                          Definition,
                                          Caption_Status)   

                                          values(%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)"""
        values = (
            row["channel_Name"],
            row["channel_Id"],
            row["video_Id"],
            row["Title"],
            row["Tags"],
            row["Thumbnail"],
            row["Description"],
            row["PublishedAt"],
            row["Duration"],
            row["Views"],
            row["Likes"],
            row["Comments"],
            row["Favorite_Count"],
            row["Definition"],
            row["Caption_Status"],
        )

        cursor.execute(insert_query, values)
        mydb.commit()

In [55]:
videos_table("Priyanka Deshpande")

In [60]:
# comment table
def comment_table(channel_name_s):
    mydb = psycopg2.connect(
        host="localhost",
        user="postgres",
        password="password",
            database="databasename",
        port="5432",
    )
    cursor = mydb.cursor()



    create_query = """create table if not exists comments(comment_Id varchar(100),
                                                            video_Id varchar(50),
                                                            Comment_Text text,
                                                            Comment_Author varchar(150),
                                                            Comment_Published timestamp )"""

    cursor.execute(create_query)
    mydb.commit()

    single_comment_details=[]
    db = client["youtube_data"]
    coll1 = db["channel_details"]
    for ch_data in coll1.find({"channel_information.channel_name":"Madan Gowri"}, {"_id": 0,}):
            single_comment_details.append(ch_data["comment_information"])

    df_single_comment_details=pd.DataFrame(single_comment_details[0])

    for index, row in df_single_comment_details.iterrows():
        insert_query = """insert into comments(comment_Id,
                                              video_Id,
                                              Comment_Text,
                                              Comment_Author,
                                              Comment_Published)
                                                            
                                            values(%s,%s,%s,%s,%s)"""
        values = (
            row["comment_Id"],
            row["video_Id"],
            row["Comment_Text"],
            row["Comment_Author"],
            row["Comment_Published"],
        )

        cursor.execute(insert_query, values)
        mydb.commit()

In [58]:
single_comment_details=[]
db = client["youtube_data"]
coll1 = db["channel_details"]
for ch_data in coll1.find({"channel_information.channel_name":"Vj Siddhu Vlogs"}, {"_id": 0,}):
        single_comment_details.append(ch_data["comment_information"])

df_single_comment_details=pd.DataFrame(single_comment_details[0])

In [61]:
df_single_comment_details

,comment_Id,video_Id,Comment_Text,Comment_Author,Comment_Published
0,Ugx-8ylxckE4Qt8HCnd4AaABAg,Bi_Ot8_yDpo,"<a href=""https://www.youtube.com/watch?v=Bi_Ot...",@tastyfoodjournal,2024-05-22T17:34:34Z
1,UgwCaX6N8YzBgkpKH_14AaABAg,Bi_Ot8_yDpo,😂😂,@jacksparrowgaming3097,2024-05-22T17:33:19Z
2,UgxoehW9vjAyzqvObc54AaABAg,Bi_Ot8_yDpo,Highlight yh Harshath dan paaaa.. vera level fun,@zipporahsimon4076,2024-05-22T17:33:16Z
3,UgzHodV8NLHJXANSdnV4AaABAg,Bi_Ot8_yDpo,Last 3 episode layea ithu tha worth👍🤟,@parvejrahman8244,2024-05-22T17:32:29Z
4,Ugx_EBykRuVjZd_n9x54AaABAg,Bi_Ot8_yDpo,Korea bore a iruka illa video va😂😂,@flee2313,2024-05-22T17:32:23Z
...,...,...,...,...,...
2494,UgzpQafFclLxtefuU3R4AaABAg,6ZX8gywf6sQ,bro sri lanka and kolkata series movie poduge ...,@satishu916,2024-03-08T02:05:30Z
2495,UgyiQ1UqcMY1tiILdIp4AaABAg,6ZX8gywf6sQ,Audio clear ra kakela ❤,@pugazhendhi5747,2024-03-07T13:27:50Z
2496,Ugw36VGZSaP9f2l8wYh4AaABAg,6ZX8gywf6sQ,Bro lv u u r 10 yrs fan bro❤,@laxmanln3325,2024-03-07T07:52:53Z
2497,UgxdQHIsHOP-eoCrZ9x4AaABAg,6ZX8gywf6sQ,anaadhai illam kku donate pannittu irundheenga...,@suryas9996,2024-03-07T05:43:31Z


In [62]:
def tables():
    channel_table()
    playlist_table()
    videos_table()
    comment_table()

    return "Tables Created Successfully"

In [63]:
Tables=tables()

TypeError: channel_table() missing 1 required positional argument: 'channel_name_s'

In [ ]:
Tables

In [64]:
def show_channel_table():

  ch_list=[]
  db=client["youtube_data"]
  coll1=db["channel_details"]
  for ch_data in coll1.find({},{"_id":0,"channel_information":1}):
    ch_list.append(ch_data["channel_information"])
  df=st.dataframe(ch_list)

  return df

In [65]:
def show_playlist_table():
    pl_list=[]
    db=client["youtube_data"]
    coll1=db["channel_details"]
    for pl_data in coll1.find({},{"_id":0,"playlist_information":1}):
      for i in range(len(pl_data["playlist_information"])):
        pl_list.append(pl_data["playlist_information"][i])

    df1=st.dataframe(pl_list)

    return df1

In [66]:
def show_video_table():
    vi_list=[]
    db=client["youtube_data"]
    coll1=db["channel_details"]
    for vi_data in coll1.find({},{"_id":0,"video_information":1}):
        for i in range(len(vi_data["video_information"])):
          vi_list.append(vi_data["video_information"][i])

    df2=st.dataframe(vi_list)

    return df2

In [67]:
def show_comment_table():

  com_list=[]
  db=client["youtube_data"]
  coll1=db["channel_details"]
  for com_data in coll1.find({},{"_id":0,"comment_information":1}):
      for i in range(len(com_data["comment_information"])):
        com_list.append(com_data["comment_information"][i])

  df3=st.dataframe(com_list)

  return df3

In [68]:
#streamlit part

with st.sidebar:
    st.title(":blue[YOUTUBE DATA HAVERSTING AND WAREOUSING]")
    st.header("INDEX")
    st.header("Python Scripting")
    st.caption("Data Collection")
    st.caption("MongoDB")
    st.caption("API Integration")
    st.caption("Data Management using MongoDB and SQL")

channel_id=st.text_input("Enter The Channel ID")

if st.button("collect and store data"):
    ch_ids=[]
    db=client["Youtube_data"]
    coll1=db["channel_details"]
    for ch_data in coll1.find({},{-id:0,"channel_information":1}):
        ch_ids.append(ch_data["channel_information"]["channel_ID"])

    if channel_id in ch_ids:
        st.success("Channel Details of te given channel id already exists")
    else:
         insert=channel_details(channel_id)
         st.success(insert)


if st.button("Migrate to sql"):
      Table=tables()
      st.success(Table)

show_table=st.radio("SELECT TE TABLE FOR VIEW",("CHANNEL","PLAYLISTS","VIDEOS","COMMENTS"))

if show_table=="CHANNELS":
    show_channel_table()

elif show_table=="PLAYLISTS":
    show_playlist_table()

elif show_table=="VIDEOS":
    show_video_table()

elif show_table=="COMMENTS":
    show_comment_table()


2026-05-18 00:28:22.066 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-18 00:28:22.193 
  command:

    streamlit run C:\Users\vaide\AppData\Roaming\Python\Python314\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-05-18 00:28:22.196 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-18 00:28:22.198 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-18 00:28:22.203 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-18 00:28:22.204 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-18 00:28:22.207 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-18 00:28:22.209 Thre

In [69]:
#SQL Connection

mydb = psycopg2.connect(
        host="localhost",
        user="postgres",
        password="password",
            database="databasename",
        port="5432"
    )
cursor = mydb.cursor()

question=st.selectbox("Select Your Question",("1. All the videos and the channel name",
                                               "2. Channels with most number of videos",
                                               "3. 10 most views videos",
                                               "4. comments in each videos",
                                               "5. videos with higest likes",
                                               "6. likes of all videos",
                                               "7. view of each channel",
                                               "8. videos published in the year of 2022",
                                               "9. average duration of all in each channel",
                                               "10. videos with highest number of comments"))

if question=="1. All the videos and the channel name":
    query1='''select title as video,channel_name as channelname from videos'''
    cursor.execute(query1)
    mydb.commit()
    t1=cursor.fetchall()
    df=pd.DataFrame(t1,columns=["video title","channel name"])
    st.write(df)

elif question=="2. Channels with most number of videos":
    query2='''select channel_name as channelname,total_videos as no_videos from channels
                order by total_videos desc'''
    cursor.execute(query2)
    mydb.commit()
    t2=cursor.fetchall()
    df2=pd.DataFrame(t2,columns=["channel name","no of videos"])
    st.write(df2)

elif question=="3. 10 most views videos":
    query3='''select views as views,channel_name as channelname,title as videotitle from videos
                where views is not null order by views desc limit 10 '''
    cursor.execute(query3)
    mydb.commit()
    t3=cursor.fetchall()
    df3=pd.DataFrame(t3,columns=["views","channel name","videotitle"])
    st.write(df3)


elif question=="4. comments in each videos":
    query4='''select comments as No_comments,title as videotitle from videos
                where comments is not null'''
    cursor.execute(query4)
    mydb.commit()
    t4=cursor.fetchall()
    df4=pd.DataFrame(t4,columns=["no of comments","videotitle"])
    st.write(df4)


elif question=="5. videos with higest likes":
    query5='''select title as titlevideos,channel_name as channelname,likes as likecount from videos
                where likes is not null order by likes desc'''
    cursor.execute(query5)
    mydb.commit()
    t5=cursor.fetchall()
    df5=pd.DataFrame(t5,columns=["videotitle","channelname","likescount"])
    st.write(df5)


elif question=="6. likes of all videos":
    query6='''select likes as likecount, title as videotitle from videos'''
    cursor.execute(query6)
    mydb.commit()
    t6=cursor.fetchall()
    df6=pd.DataFrame(t6,columns=["likescount","videotitle"])
    st.write(df6)


elif question== "7. view of each channel":
    query7='''select channel_name as channelname , views as totalviews from channels'''
    cursor.execute(query7)
    mydb.commit()
    t7=cursor.fetchall()
    df7=pd.DataFrame(t7,columns=["channel name","totalviews"])
    st.write(df7)

if question=="8. videos published in the year of 2022":
    query8='''select title as videotitle, published_date as publishedAt, channel_name as channelname from videos
            where extract (year from published_date)=2022'''
    cursor.execute(query8)
    mydb.commit()
    t8=cursor.fetchall()
    df8=pd.DataFrame(t8,columns=["videotitle","published_date","channelname"])
    st.write(df8)

elif question=="9. average duration of all in each channel":
    query9='''select channel_name as channelname, AVG(duration) as averageduration from videos group by channel_name'''
    cursor.execute(query9)
    mydb.commit()
    t9=cursor.fetchall()
    df9=pd.DataFrame(t9,columns=["channelname","averageduration"])
    st.write(df9)

    T9=[]
    for index,row in df9.iterrows():
        channel_title=row["channelname"]
        average_duration=row['averageduration']
        average_duration_str=str (average_duration)
        T9.append(dict(channeltitle=channel_title,avgduration=average_duration_str))
    df1=pd.DataFrame(T9)
    st.write(df1)


elif question=="10. videos with highest number of comments":
        query10='''select title as videotitle,channel_name as channelname,comments as comments from videos
                where comments is not null order by comments desc'''
        cursor.execute(query10)
        mydb.commit()
        t10=cursor.fetchall()
        df10=pd.DataFrame(t10,columns=["video title","channel name","comments"])
        st.write(df10)

2026-05-18 00:28:40.872 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-18 00:28:40.874 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-18 00:28:40.876 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-18 00:28:40.877 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-18 00:28:40.879 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-18 00:28:40.880 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-18 00:28:40.881 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-18 00:28:40.896 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [70]:
mydb = psycopg2.connect(
        host="localhost",
        user="postgres",
        password="password",
            database="databasename",
        port="5432"
    )
cursor = mydb.cursor()


if question=="10. videos with highest number of comments":
        query10='''select title as videotitle,channel_name as channelname,comments as comments from videos
                where comments is not null order by comments desc'''
        cursor.execute(query10)
        mydb.commit()
        t10=cursor.fetchall()
        df10=pd.DataFrame(t10,columns=["video title","channel name","comments"])
        st.write(df9)